In [1]:
import os
import numpy as np
import pandas as pd
import pandas_market_calendars as mcal
import datetime as dt
from data_processor import DataReader, DataPrep
from scipy import stats

In [2]:
nyse = mcal.get_calendar('NYSE')
# Get holidays
holidays = nyse.holidays().holidays

### EDA

In [3]:
daily_data_path = r'data/daily_data'
intraday_data_path = r'data/intraday_data'

In [4]:
reader = DataReader()
intraday_df = reader.read_intraday_data(intraday_data_path)
daily_df = reader.read_daily_data(daily_data_path)
intraday_df.dropna(subset= 'CumReturnResid', inplace=True)

In [10]:
query_date = dt.date(2010, 1, 4)
query_id = 'BBG000MQ1SN9'

In [39]:
def RSI(intraday_return):
    
    returns = intraday_return.iloc[:-2].CumReturnResid.diff()
    gains = returns.where(returns > 0, 0)
    losses = returns.where(returns < 0, 0)
    # Calculate the average gain and loss over the period (14 days in this example)
    average_gain = gains.rolling(window=14, min_periods=1).mean().iloc[-1]
    average_loss = losses.rolling(window=14, min_periods=1).mean().iloc[-1]
    
    # Calculate RS and RSI
    RS = average_gain / average_loss
    RSI = 100 - (100 / (1 + RS))
    
    return RSI


def calculate_rsi_vectorized(data, column='CumReturnResid', period=14):
    delta = data[column].iloc[:-2].diff()
    gains = np.maximum(delta, 0)
    losses = np.abs(np.minimum(delta, 0))

    # Calculate the Exponential Moving Average (EMA) of gains and losses
    avg_gain = gains.ewm(com=period-1, min_periods=period).mean()
    avg_loss = losses.ewm(com=period-1, min_periods=period).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    
    return rsi

In [40]:
daily_RSI = intraday_df.groupby(['Date', 'Id']).apply(calculate_rsi_vectorized)

In [ ]:
daily_df.eval('Range = abs(Close - Open)', inplace= True)

In [ ]:
daily_df.query('SharesAdjFactor > 1')

In [ ]:
daily_df.groupby('Id').apply(lambda x: x.Range.shift().corr(x.EST_VOL)).describe()

In [ ]:
def query_id(df):
    
    df = df[df.CumReturnResid.isna()]
    df.drop_duplicates('Date', inplace= True)
    df['Diff'] = df.Date.diff().dt.days
    return df[['Date', 'Diff']].sort_values('Diff')

In [ ]:
test_df = intraday_df.groupby('Id').apply(query_id)

In [ ]:
test_df = intraday_df.query('Id == "BBG000BLM0V1"')#.drop_duplicates('Date').reset_index(drop= True).iloc[680:690]#query('Date == @query_date').copy()
test_df[test_df.CumReturnResid.isna()].drop_duplicates('Date').reset_index(drop= True).iloc[60:66]#query('Date == @query_date').copy()

In [ ]:
data_prep = DataPrep(intraday_df, daily_df)

In [ ]:
target_df = data_prep.get_target(clip_MAD=True, normalize= True)

In [ ]:
target_df

### ID with different names

### Features Prep

In [ ]:
daily_df